<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula03a%20-%20valida%C3%A7%C3%A3o%20cruzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy: {accuracy_score(y_test, y_pred)}")

accuracy: 0.9444444444444444


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy: {accuracy_score(y_test, y_pred)}")

accuracy: 0.9722222222222222


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy k= 5: {accuracy_score(y_test, y_pred)}")

model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"accuracy k=15: {accuracy_score(y_test, y_pred)}")

accuracy k= 5: 1.0
accuracy k=15: 0.9166666666666666


In [31]:
from pprint import pprint

models = [
    ("K= 5", KNeighborsClassifier(n_neighbors=5)),
    ("K=15", KNeighborsClassifier(n_neighbors=15))
]

def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

def evaluate_models(models, X, y, scores={}):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    for model_name, model in models:
        acc = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)
        if model_name not in scores:
            scores[model_name] = []
        scores[model_name].append(acc)
    return scores

def multiple_holdouts(models, X, y, n_repeats=50):
    scores = {}
    for _ in range(n_repeats):
        scores = evaluate_models(models, X, y, scores)
    return scores

scores = multiple_holdouts(models, X, y)
for model_name, scores in scores.items():
    print(f"{model_name}: {sum(scores) / len(scores)}")

K= 5: 0.9622222222222221
K=15: 0.9638888888888889


In [42]:
import numpy as np

def cross_validation(models, X, y, n_folds=5, scores={}):
    idxs = np.random.permutation(len(y))
    test_size = len(y) // n_folds
    for i in range(n_folds):
        test_idxs = idxs[i*test_size:(i+1)*test_size]
        X_test, y_test = X[test_idxs], y[test_idxs]
        train_idxs = np.concatenate((idxs[:i*test_size], idxs[(i+1)*test_size:]))
        X_train, y_train = X[train_idxs], y[train_idxs]
        scaler = StandardScaler()
        scaler.fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        for model_name, model in models:
            acc = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test)
            if model_name not in scores:
                scores[model_name] = []
            scores[model_name].append(acc)
    return scores

scores = cross_validation(models, X, y)
for model_name, scores in scores.items():
    print(f"{model_name}: {sum(scores) / len(scores)}")

K= 5: 0.9542857142857143
K=15: 0.9542857142857143


In [45]:
def leave_one_out(model, X, y, scores={}):
  k = len(y)
  return cross_validation(model, X, y, k, scores)

scores = leave_one_out(models, X, y)
for model_name, scores in scores.items():
    print(f"{model_name}: {sum(scores) / len(scores)}")

K= 5: 0.9719101123595506
K=15: 0.9662921348314607


In [50]:
from sklearn.model_selection import cross_validate

model = KNeighborsClassifier()

scores = cross_validate(model, X, y)
pprint(scores)

{'fit_time': array([0.00809193, 0.00671673, 0.0067687 , 0.00581884, 0.0058434 ]),
 'score_time': array([0.02274275, 0.01404929, 0.01505136, 0.01493382, 0.0077455 ]),
 'test_score': array([0.72222222, 0.66666667, 0.63888889, 0.65714286, 0.77142857])}


In [51]:
scores = cross_validate(model, X, y)
print(scores['test_score'].mean())

0.6912698412698413


In [67]:
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin

class CustomPipeline(BaseEstimator, ClassifierMixin, TransformerMixin):
    def __init__(self, steps):
      self.steps = steps
    def fit(self, X, y):
      T = X
      for i in range(len(self.steps)):
        if hasattr(self.steps[i], "fit_transform"):
          T = self.steps[i].fit_transform(T, y)
        else:
          T = self.steps[i].fit(T, y)
      return self
    def predict(self, X):
      T = X
      for i in range(len(self.steps)):
        if hasattr(self.steps[i], "transform"):
          T = self.steps[i].transform(T)
        else:
          T = self.steps[i].predict(T)
      return T

pipe = CustomPipeline([
    StandardScaler(),
    KNeighborsClassifier()
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print("accuracy", accuracy_score(y_test, y_pred))

accuracy 0.9444444444444444


In [74]:
scores = cross_validate(pipe, X, y)
print(scores['test_score'].mean())

0.8934920634920636


In [76]:
scores = cross_validate(pipe, X, y, cv=5)
print(scores['test_score'].mean())

0.8934920634920636


In [78]:
from sklearn.model_selection import KFold

spliter = KFold(n_splits=5)
scores = cross_validate(pipe, X, y, cv=spliter)
print(scores['test_score'].mean())

0.8934920634920636


In [80]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2])

In [96]:
spliter = KFold(n_splits=5, shuffle=True)
scores = cross_validate(pipe, X, y, cv=spliter)
print(scores['test_score'].mean())

0.9661904761904763


In [104]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier()
)
spliter = KFold(n_splits=5, shuffle=True)
scores = cross_validate(pipe, X, y, cv=spliter)
print(scores['test_score'].mean())

0.9550793650793651


In [118]:
from sklearn.model_selection import LeaveOneOut

spliter = LeaveOneOut()
scores = cross_validate(pipe, X, y, cv=spliter)
print(scores['test_score'].mean())
print(scores['test_score'])

0.9719101123595506
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.
 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [119]:
from sklearn.model_selection import RepeatedKFold

spliter = RepeatedKFold(n_splits=5, n_repeats=10)
scores = cross_validate(pipe, X, y, cv=spliter)
print(scores['test_score'].mean())
print(scores['test_score'])

0.964015873015873
[0.94444444 0.97222222 1.         1.         0.91428571 0.94444444
 0.94444444 0.94444444 0.97142857 1.         0.94444444 0.94444444
 0.97222222 0.97142857 1.         1.         0.97222222 0.97222222
 1.         0.88571429 1.         0.91666667 0.97222222 0.91428571
 0.97142857 1.         0.94444444 0.94444444 0.97142857 1.
 0.97222222 0.94444444 0.94444444 0.91428571 0.97142857 0.97222222
 1.         0.91666667 0.91428571 1.         0.97222222 0.97222222
 1.         0.94285714 1.         1.         1.         0.94444444
 0.91428571 0.97142857]
